# RelCheck × RelTR: Scene-Graph-Grounded Relation Correction

**Architecture:**
1. Upload image → RelTR builds a scene graph (objects + relations) in one forward pass
2. Scene graph serialized → TF-IDF vector store (lightweight, no API)
3. RelCheck verifier flags `(subject, wrong_rel, object)` as hallucinated
4. Query scene graph → retrieve correct relation from RelTR's visual classifier
5. Llama-3.3-70B does the surgical text edit: replace only the wrong span

**Why defensible for the paper:** RelTR classifies over 51 VG predicate classes from visual features — it is not a language model and cannot hallucinate relations.

**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7**

In [ ]:
# ── CELL 1: Install RelTR + download pretrained weights ──────────────────────
import os, subprocess, sys

if not os.path.exists('RelTR'):
    !git clone https://github.com/yrcong/RelTR.git
else:
    print('RelTR already cloned')

!pip install -q matplotlib scipy scikit-learn

os.makedirs('RelTR/ckpt', exist_ok=True)
if not os.path.exists('RelTR/ckpt/checkpoint0149.pth'):
    print('Downloading pretrained RelTR weights...')
    !pip install -q gdown
    result = subprocess.run(
        [sys.executable, '-m', 'gdown',
         'https://drive.google.com/uc?id=1F_B4v6oqKpXKdD9YGz2qGZFsGQFDL5JY',
         '-O', 'RelTR/ckpt/checkpoint0149.pth'],
        capture_output=True, text=True
    )
    if not os.path.exists('RelTR/ckpt/checkpoint0149.pth') or \
       os.path.getsize('RelTR/ckpt/checkpoint0149.pth') < 1_000_000:
        print('gdown failed — upload checkpoint0149.pth manually to RelTR/ckpt/ via Files panel')
    else:
        print(f"Downloaded: {os.path.getsize('RelTR/ckpt/checkpoint0149.pth')/1e6:.1f} MB")
else:
    print(f"Weights present ({os.path.getsize('RelTR/ckpt/checkpoint0149.pth')/1e6:.1f} MB)")
print('Done.')

In [ ]:
# ── CELL 2: Upload your image ─────────────────────────────────────────────────
from google.colab import files
from PIL import Image
import shutil, os

uploaded = files.upload()
img_path = list(uploaded.keys())[0]
os.makedirs('RelTR/images', exist_ok=True)
shutil.copy(img_path, f'RelTR/images/{img_path}')

test_image = Image.open(img_path).convert('RGB')
print(f'Image: {img_path}  size={test_image.size}')
display(test_image)

In [ ]:
# ── CELL 3: Run RelTR inference → scene graph ────────────────────────────────
import sys, os
sys.path.insert(0, 'RelTR')

import torch
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import argparse
from models import build_model

CLASSES = [
    'N/A', 'airplane', 'animal', 'arm', 'bag', 'banana', 'basket', 'beach',
    'bear', 'bed', 'bench', 'bike', 'bird', 'board', 'boat', 'book', 'boot',
    'bottle', 'bowl', 'box', 'boy', 'branch', 'building', 'bus', 'cabinet',
    'cap', 'car', 'cat', 'chair', 'child', 'clock', 'coat', 'counter', 'cow',
    'cup', 'curtain', 'desk', 'dog', 'door', 'drawer', 'ear', 'elephant',
    'engine', 'eye', 'face', 'fence', 'finger', 'flag', 'flower', 'food',
    'fork', 'fruit', 'giraffe', 'girl', 'glass', 'glove', 'guy', 'hair',
    'hand', 'handle', 'hat', 'head', 'helmet', 'hill', 'horse', 'house',
    'jacket', 'jean', 'kid', 'kite', 'lady', 'lamp', 'laptop', 'leaf',
    'leg', 'letter', 'light', 'logo', 'man', 'men', 'motorcycle', 'mountain',
    'mouth', 'neck', 'nose', 'number', 'orange', 'pant', 'paper', 'paw',
    'people', 'person', 'phone', 'pillow', 'pizza', 'plane', 'plant', 'plate',
    'player', 'pole', 'post', 'pot', 'racket', 'railing', 'rock', 'roof',
    'room', 'screen', 'seat', 'sheep', 'shelf', 'shirt', 'shoe', 'short',
    'sidewalk', 'sign', 'sink', 'skateboard', 'ski', 'skier', 'sleeve',
    'snow', 'sock', 'stand', 'street', 'surfboard', 'table', 'tail', 'tie',
    'tile', 'tire', 'toilet', 'towel', 'tower', 'track', 'train', 'tree',
    'truck', 'trunk', 'umbrella', 'vase', 'vegetable', 'vehicle', 'wave',
    'wheel', 'window', 'windshield', 'wing', 'wire', 'woman', 'wood'
]

REL_CLASSES = [
    'N/A', 'above', 'across', 'against', 'along', 'and', 'at', 'attached to',
    'behind', 'belonging to', 'between', 'carrying', 'covered in', 'covering',
    'eating', 'flying in', 'for', 'from', 'growing on', 'hanging from', 'has',
    'holding', 'in', 'in front of', 'laying on', 'looking at', 'lying on',
    'made of', 'mounted on', 'near', 'of', 'on', 'on back of', 'over',
    'painted on', 'parked on', 'part of', 'playing', 'riding', 'says',
    'sitting on', 'standing on', 'to', 'under', 'using', 'walking in',
    'walking on', 'watching', 'wearing', 'wears', 'with'
]

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    return torch.stack([(x_c-0.5*w),(y_c-0.5*h),(x_c+0.5*w),(y_c+0.5*h)], dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)
    return b

# ── Build + load model ────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

args = argparse.Namespace(
    # backbone
    backbone='resnet50',
    dilation=False,
    lr_backbone=0,
    return_interm_layers=False,
    # transformer
    position_embedding='sine',
    enc_layers=6,
    dec_layers=6,
    dim_feedforward=2048,
    hidden_dim=256,
    dropout=0.1,
    nheads=8,
    num_entities=100,
    num_triplets=200,
    pre_norm=False,
    # loss weights (Hungarian matcher)
    set_cost_class=1,
    set_cost_bbox=5,
    set_cost_giou=2,
    set_iou_threshold=0.7,
    # loss coefficients
    bbox_loss_coef=5,
    giou_loss_coef=2,
    rel_loss_coef=1,
    eos_coef=0.1,
    # misc
    aux_loss=False,
    dataset='vg',
    device=device,
)

model, _, _ = build_model(args)
print('Model architecture built ✅')

ckpt_path = 'RelTR/ckpt/checkpoint0149.pth'
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f'Checkpoint not found. Upload it to {ckpt_path}')

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval().to(device)
print(f'Weights loaded ✅  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)')

# ── Inference ─────────────────────────────────────────────────────────────────
transform = T.Compose([
    T.Resize(800), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

img = Image.open(img_path).convert('RGB')
img_t = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(img_t)

# ── Parse predictions ─────────────────────────────────────────────────────────
CONF_THRESHOLD = 0.3

probas     = outputs['rel_logits'].softmax(-1)[0, :, :-1]
probas_sub = outputs['sub_logits'].softmax(-1)[0, :, :-1]
probas_obj = outputs['obj_logits'].softmax(-1)[0, :, :-1]

keep = torch.nonzero(
    (probas.max(-1)[0] > CONF_THRESHOLD) &
    (probas_sub.max(-1)[0] > CONF_THRESHOLD) &
    (probas_obj.max(-1)[0] > CONF_THRESHOLD)
).flatten()

print(f'\n{len(keep)} triples pass threshold={CONF_THRESHOLD}')
if len(keep) == 0:
    print('⚠️  No triples detected. Try lowering CONF_THRESHOLD to 0.15.')

try:
    raw_bboxes_sub = rescale_bboxes(outputs['sub_boxes'][0, keep], img.size)
    raw_bboxes_obj = rescale_bboxes(outputs['obj_boxes'][0, keep], img.size)
    has_boxes = True
except KeyError:
    raw_bboxes_sub = raw_bboxes_obj = None
    has_boxes = False
    print('Note: bbox keys not found — visualization will be skipped')

# Build scene_graph with bboxes embedded (survives any reordering)
scene_graph = []
for i, idx in enumerate(keep):
    entry = {
        'subject':        CLASSES[probas_sub[idx].argmax()],
        'predicate':      REL_CLASSES[probas[idx].argmax()],
        'object':         CLASSES[probas_obj[idx].argmax()],
        'subject_conf':   probas_sub[idx].max().item(),
        'predicate_conf': probas[idx].max().item(),
        'object_conf':    probas_obj[idx].max().item(),
    }
    if has_boxes:
        entry['bbox_sub'] = raw_bboxes_sub[i].tolist()
        entry['bbox_obj'] = raw_bboxes_obj[i].tolist()
    scene_graph.append(entry)

scene_graph.sort(key=lambda x: -x['predicate_conf'])

print(f'\nScene graph: {len(scene_graph)} triples\n')
print(f'{"Subject":<15} {"Predicate":<22} {"Object":<15} {"PConf":>6}')
print('-' * 62)
for t in scene_graph:
    print(f"{t['subject']:<15} {t['predicate']:<22} {t['object']:<15} {t['predicate_conf']:>6.2f}")

In [ ]:
# ── CELL 4: Build vector store ────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def build_chunks(scene_graph):
    if not scene_graph:
        return []
    chunks = []
    by_subject = {}
    for t in scene_graph:
        by_subject.setdefault(t['subject'], []).append(f"{t['predicate']} {t['object']}")
    for subj, rels in by_subject.items():
        chunks.append({'text': f"{subj}: {', '.join(rels)}", 'subject': subj, 'type': 'summary'})
    for t in scene_graph:
        chunks.append({
            'text': f"{t['subject']} {t['predicate']} {t['object']}",
            'subject': t['subject'], 'predicate': t['predicate'],
            'object': t['object'], 'conf': t['predicate_conf'], 'type': 'triple'
        })
    return chunks

class SceneGraphVectorStore:
    def __init__(self, chunks):
        self.chunks = chunks
        if not chunks:
            self.empty = True
            return
        self.empty = False
        self.texts = [c['text'] for c in chunks]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform(self.texts)

    def retrieve(self, query, top_k=5):
        if self.empty:
            return []
        q_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(q_vec, self.matrix).flatten()
        top_idx = sims.argsort()[::-1][:top_k]
        return [(self.chunks[i], sims[i]) for i in top_idx if sims[i] > 0]

chunks = build_chunks(scene_graph)
store = SceneGraphVectorStore(chunks)

if store.empty:
    print('⚠️  Empty scene graph — vector store has no chunks. Lower CONF_THRESHOLD in Cell 3.')
else:
    print(f'Vector store: {len(chunks)} chunks')
    for c in chunks:
        print(f'  "{c["text"]}"')

In [ ]:
# ── CELL 5: Core lookup + correction functions ────────────────────────────────

# Synonym map for entity matching
# (BLIP-2 might say 'man', RelTR might detect 'person' — treat as same)
SYNONYMS = {
    'man': 'person', 'woman': 'person', 'boy': 'person', 'girl': 'person',
    'guy': 'person', 'lady': 'person', 'kid': 'child',
    'bicycle': 'bike', 'sofa': 'couch', 'motorbike': 'motorcycle',
    'aeroplane': 'airplane', 'tv': 'screen', 'monitor': 'screen',
}

def normalize_entity(e):
    """Lowercase + apply synonym map."""
    e = e.strip().lower()
    return SYNONYMS.get(e, e)

def entities_match(a, b):
    """Check if two entity strings refer to the same thing."""
    a, b = normalize_entity(a), normalize_entity(b)
    return a == b or a in b or b in a


def lookup_reltr(scene_graph, subject, obj):
    """
    Stage 1: Direct entity-pair lookup in RelTR scene graph.
    Returns list of matching triples sorted by predicate confidence.
    """
    matches = []
    for t in scene_graph:
        if entities_match(subject, t['subject']) and entities_match(obj, t['object']):
            matches.append(t)
        # Also check reversed direction (RelTR may flip subject/object)
        elif entities_match(subject, t['object']) and entities_match(obj, t['subject']):
            # Return reversed so caller gets (subject, predicate, object) direction
            reversed_t = dict(t)
            reversed_t['subject'] = t['object']
            reversed_t['object']  = t['subject']
            reversed_t['_reversed'] = True
            matches.append(reversed_t)
    matches.sort(key=lambda x: -x['predicate_conf'])
    return matches


def graph_rag_correct(store, scene_graph, subject, hallucinated_rel, obj, verbose=True):
    """
    Full correction lookup:
    Stage 1 → direct entity-pair match in scene graph
    Stage 2 → TF-IDF vector retrieval fallback
    Returns dict: {correction, confidence, source} or {correction: 'DELETE'}
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Input triple: ({subject}, {hallucinated_rel}, {obj})")
        print(f"{'='*60}")

    # Stage 1 — direct lookup
    if verbose:
        print('\n[Stage 1] Direct entity-pair lookup...')
    matches = lookup_reltr(scene_graph, subject, obj)

    for m in matches:
        rev = ' (reversed)' if m.get('_reversed') else ''
        if verbose:
            print(f"  Match{rev}: ({m['subject']}, {m['predicate']}, {m['object']}) "
                  f"conf={m['predicate_conf']:.2f}")

    if matches:
        best = matches[0]
        if best['predicate'] == hallucinated_rel:
            if verbose:
                print(f"  ⚠  RelTR agrees with '{hallucinated_rel}' — triple may not be hallucinated")
            return {'correction': hallucinated_rel, 'note': 'RelTR agrees', 'source': 'RelTR direct'}
        if verbose:
            print(f"\n  ✅ CORRECTION: '{hallucinated_rel}' → '{best['predicate']}' "
                  f"(conf={best['predicate_conf']:.2f})")
        return {
            'correction': best['predicate'],
            'confidence': best['predicate_conf'],
            'source': 'RelTR direct'
        }

    # Stage 2 — vector retrieval fallback
    if verbose:
        print('\n[Stage 2] Vector retrieval fallback...')
    query = f"{subject} {obj}"
    retrieved = store.retrieve(query, top_k=5)
    for chunk, score in retrieved:
        if verbose:
            print(f"  sim={score:.3f} | {chunk['text']}")
    for chunk, score in retrieved:
        if chunk.get('type') == 'triple' and score > 0.1:
            pred = chunk.get('predicate', '')
            if pred and pred != hallucinated_rel:
                if verbose:
                    print(f"  ✅ CORRECTION (retrieval): '{hallucinated_rel}' → '{pred}'")
                return {'correction': pred, 'similarity': score, 'source': 'vector retrieval'}

    if verbose:
        print(f"  ❌ No match found → recommend DELETE")
    return {'correction': 'DELETE', 'reason': 'entity pair not in RelTR scene graph'}


def apply_caption_correction(caption, subject, hallucinated_rel, obj, correct_rel):
    """
    Surgically replace the hallucinated relation span in the caption.
    Strategy: simple string replacement of the relation phrase.
    For DELETE: remove the entire clause containing the hallucinated triple.
    """
    import re

    if correct_rel == 'DELETE':
        # Remove the clause: find sentence containing both subject + hallucinated_rel
        sentences = re.split(r'(?<=[.!?])\s+', caption)
        kept = []
        for s in sentences:
            if hallucinated_rel.lower() in s.lower() and subject.lower() in s.lower():
                continue  # drop this sentence
            kept.append(s)
        corrected = ' '.join(kept).strip()
        return corrected if corrected else caption  # fallback: keep original
    else:
        # Replace the relation phrase
        corrected = re.sub(
            re.escape(hallucinated_rel),
            correct_rel,
            caption,
            flags=re.IGNORECASE
        )
        return corrected


print('Functions ready: graph_rag_correct(), apply_caption_correction()')

In [ ]:
# ── CELL 6: End-to-end demo ───────────────────────────────────────────────────
# Fill in these three inputs based on your image and caption:

CAPTION          = "A man standing next to a horse in the field."
SUBJECT          = "man"
OBJECT           = "horse"
HALLUCINATED_REL = "standing next to"

# ── Step 1: Look up correct relation from RelTR scene graph ───────────────────
result = graph_rag_correct(
    store=store, scene_graph=scene_graph,
    subject=SUBJECT, hallucinated_rel=HALLUCINATED_REL, obj=OBJECT
)

# ── Step 2: Apply correction to caption ───────────────────────────────────────
corrected_caption = apply_caption_correction(
    caption=CAPTION,
    subject=SUBJECT,
    hallucinated_rel=HALLUCINATED_REL,
    obj=OBJECT,
    correct_rel=result['correction']
)

print(f"\n{'─'*60}")
print(f"ORIGINAL:  {CAPTION}")
print(f"CORRECTED: {corrected_caption}")
print(f"SOURCE:    {result.get('source', 'N/A')}")
if 'confidence' in result:
    print(f"CONF:      {result['confidence']:.2f}")
print(f"{'─'*60}")

In [ ]:
# ── CELL 7: Visualize scene graph bboxes on image ────────────────────────────
# Bboxes are stored inside each scene_graph entry (no ordering bug)

if not has_boxes:
    print('(Visualization skipped — no bbox data in model output)')
elif not scene_graph:
    print('(Visualization skipped — empty scene graph)')
else:
    fig, ax = plt.subplots(1, 1, figsize=(13, 9))
    ax.imshow(test_image)
    ax.set_title(f'RelTR Scene Graph — {len(scene_graph)} triples  |  threshold={CONF_THRESHOLD}',
                 fontsize=13)

    colors = plt.cm.tab10(np.linspace(0, 1, min(len(scene_graph), 10)))

    for i, t in enumerate(scene_graph):
        c = colors[i % len(colors)]
        label = f"{t['subject']} —{t['predicate']}→ {t['object']}  [{t['predicate_conf']:.2f}]"

        # Subject bbox (solid border)
        x1, y1, x2, y2 = t['bbox_sub']
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor=c, facecolor='none'))
        ax.text(x1, max(y1-5, 0), t['subject'], color=c, fontsize=8, weight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

        # Object bbox (dashed border)
        x1, y1, x2, y2 = t['bbox_obj']
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor=c, facecolor='none', linestyle='--'))
        ax.text(x1, min(y2+13, img.size[1]-1),
                f"→{t['predicate']}→{t['object']}",
                color=c, fontsize=7,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    ax.axis('off')
    plt.tight_layout()
    plt.savefig('scene_graph_viz.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: scene_graph_viz.png')


# ── Coverage analysis ─────────────────────────────────────────────────────────
print('\n── RelTR predicate distribution ──────────────────────────────')
from collections import Counter
pred_counts = Counter(t['predicate'] for t in scene_graph)
for pred, cnt in pred_counts.most_common():
    print(f'  {pred:<22} {cnt}')